# Лабораторная работа 3. RAG-конвейер для базы научных статей

## Подготовка

In [24]:
%matplotlib inline

import os
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

import re
import time
import random
import warnings
from collections import Counter
from pathlib import Path

import faiss
import numpy as np
import pandas as pd
import symspellpy
import torch
from anthropic import Anthropic
from dotenv import load_dotenv
from IPython.display import display
from langchain_text_splitters import RecursiveCharacterTextSplitter
from rapidfuzz import fuzz
from sentence_transformers import SentenceTransformer
from symspellpy import SymSpell, Verbosity

faiss.omp_set_num_threads(1)
warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 110)
pd.set_option("display.width", 160)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

load_dotenv(".env")
GLM_API_KEY = os.getenv("GLM_API_KEY")

DATA_PATH = Path("data/papers.csv")

## Пункт 1. Сбор корпуса документов

> Выбрать предметную область. Собрать корпус минимум из 300 статей (название, авторы, год, аннотация, ссылка). Можно взять корпус из первого задания, если он подходит по количеству. Сохранить метаданные и тексты в структурированном виде (Pandas DataFrame или JSON Lines).

**Тема: Entity Resolution** — корпус из лабораторной работы 1: **477 статей за 2020–2026 годы**, собран скриптом `fetch_arxiv.py` через arXiv API. Скрипт проверяет обязательные поля и уникальность arXiv id.

In [25]:
df = pd.read_csv(DATA_PATH, dtype=str)
df["year"] = df["published"].str[:4].astype(int)
df["url"] = "https://arxiv.org/abs/" + df["arxiv_id"]
df.head(3)[["arxiv_id", "title", "published", "authors"]]

,arxiv_id,title,published,authors
0,2001.01895,Machine-learning classifiers for logographic name matching in public health applications: approaches for i...,2020-01-07T05:21:21Z,Philip A. Collender; Zhiyue Tom Hu; Charles Li; Qu Cheng; Xintong Li; Yue You; Song Liang; Changhong Yang;...
1,2001.01961,Complexity Issues of String to Graph Approximate Matching,2020-01-07T10:40:52Z,Riccardo Dondi; Giancarlo Mauri; Italo Zoppis
2,2001.09233,Case Study: Predictive Fairness to Reduce Misdemeanor Recidivism Through Social Service Interventions,2020-01-24T23:52:55Z,Kit T. Rodolfa; Erika Salomon; Lauren Haynes; Ivan Higuera Mendieta; Jamie Larson; Rayid Ghani


Распределение статей по годам и размеры документов.

In [26]:
by_year = df["year"].value_counts().sort_index()
display(by_year.to_frame("статей"))

doc_words = (df["title"].str.strip().str.rstrip(".") + ". " + df["abstract"].str.strip()).str.split().str.len()
pd.Series({
    "документов": len(df),
    "дубликатов по названию": int(df["title"].duplicated().sum()),
    "слов на документ: среднее": round(doc_words.mean()),
    "слов на документ: медиана": int(doc_words.median()),
    "слов на документ: минимум": int(doc_words.min()),
    "слов на документ: максимум": int(doc_words.max()),
    "документов короче 150 слов, %": round((doc_words < 150).mean() * 100),
}, name="значение").to_frame()

,статей
year,
2020,45
2021,53
2022,42
2023,65
2024,71
2025,78
2026,123


,значение
документов,477
дубликатов по названию,0
слов на документ: среднее,199
слов на документ: медиана,203
слов на документ: минимум,64
слов на документ: максимум,333
"документов короче 150 слов, %",16


## Пункт 2. Препроцессинг и стратегия чанкинга (Chunking)

> Объединить название статьи и её аннотацию в единый документ для сохранения контекста. Реализовать разбиение длинных аннотаций/текстов на смысловые фрагменты: размер чанка 120–150 слов (или 500–700 символов), перекрытие 20–30 слов (или ~100 символов). Реализовать чанкинг двумя способами: на чистом Python (скользящее окно по предложениям/словам) и с использованием RecursiveCharacterTextSplitter (LangChain). К каждому чанку привязать метаданные родительской статьи (doc_id, Title, Authors, URL).

Каждая статья превращается в один документ `Название. Аннотация`. Документ режется на фрагменты двумя способами, и к каждому фрагменту привязываются метаданные родительской статьи: `doc_id`, `title`, `authors`, `year`, `url`.

In [27]:
docs = (df[["arxiv_id", "title", "authors", "year", "url", "abstract"]]
        .rename(columns={"arxiv_id": "doc_id"}).copy())
docs["text"] = docs["title"].str.strip().str.rstrip(".") + ". " + docs["abstract"].str.strip()
docs = docs[["doc_id", "title", "authors", "year", "url", "text"]]
docs.head(2)

,doc_id,title,authors,year,url,text
0,2001.01895,Machine-learning classifiers for logographic name matching in public health applications: approaches for i...,Philip A. Collender; Zhiyue Tom Hu; Charles Li; Qu Cheng; Xintong Li; Yue You; Song Liang; Changhong Yang;...,2020,https://arxiv.org/abs/2001.01895,Machine-learning classifiers for logographic name matching in public health applications: approaches for i...
1,2001.01961,Complexity Issues of String to Graph Approximate Matching,Riccardo Dondi; Giancarlo Mauri; Italo Zoppis,2020,https://arxiv.org/abs/2001.01961,Complexity Issues of String to Graph Approximate Matching. The problem of matching a query string to a dir...


Первый способ — скользящее окно

In [28]:
MAX_WORDS = 150
OVERLAP_WORDS = 25

SENT_SPLIT_RE = re.compile(r"(?<=[.!?])\s+(?=[\"'(\[]?[A-Z0-9])")


def split_sentences(text):
    parts = [s.strip() for s in SENT_SPLIT_RE.split(text) if s and s.strip()]
    return parts or [text.strip()]


def chunk_text_python(text, max_words=MAX_WORDS, overlap_words=OVERLAP_WORDS):
    units = []
    for s in split_sentences(text):
        w = s.split()
        if len(w) <= max_words:
            units.append(s)
        else:
            step = max_words - overlap_words
            for i in range(0, len(w), step):
                units.append(" ".join(w[i:i + max_words]))
                if i + max_words >= len(w):
                    break
    chunks, cur, cur_w = [], [], 0
    for s in units:
        sw = len(s.split())
        if cur and cur_w + sw > max_words:
            chunks.append(" ".join(cur))
            tail, tail_w = [], 0
            for t in reversed(cur):
                tw = len(t.split())
                if tail and tail_w + tw > overlap_words:
                    break
                tail.insert(0, t)
                tail_w += tw
            cur, cur_w = tail[:], tail_w
            if cur_w + sw > max_words:
                chunks.append(" ".join(cur))
                cur, cur_w = [], 0
        cur.append(s)
        cur_w += sw
    if cur and (not chunks or cur_w > overlap_words):
        chunks.append(" ".join(cur))
    return [c for c in chunks if c and len(c.split()) >= 5]


def build_chunks(method):
    rows = []
    for _, d in docs.iterrows():
        if method == "py":
            pieces = chunk_text_python(d["text"])
        else:
            pieces = [c.strip() for c in LC_SPLITTER.split_text(d["text"]) if len(c.strip().split()) >= 5]
        for i, p in enumerate(pieces):
            rows.append({"chunk_id": f"{d['doc_id']}::{method}::{i}", "chunk_seq": i, "text": p,
                         "doc_id": d["doc_id"], "title": d["title"], "authors": d["authors"],
                         "year": d["year"], "url": d["url"]})
    return pd.DataFrame(rows)


chunks_py = build_chunks("py")

chunks_py[["chunk_id", "doc_id", "year"]].head(2)

,chunk_id,doc_id,year
0,2001.01895::py::0,2001.01895,2020
1,2001.01895::py::1,2001.01895,2020


Второй способ — `RecursiveCharacterTextSplitter` (LangChain)

In [29]:
CHUNK_SIZE = 650
CHUNK_OVERLAP = 100
LC_SPLITTER = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", " ", ""],
)

chunks_lc = build_chunks("lc")

chunks_lc[["chunk_id", "doc_id", "year"]].head(2)

,chunk_id,doc_id,year
0,2001.01895::lc::0,2001.01895,2020
1,2001.01895::lc::1,2001.01895,2020


Сравниваем число и размер фрагментов

In [30]:
stats = []
for name, ch in [("чистый Python", chunks_py), ("LangChain", chunks_lc)]:
    nw, nc = ch["text"].str.split().str.len(), ch["text"].str.len()
    meta_cols = ["doc_id", "title", "authors", "year", "url"]
    stats.append({
        "способ": name,
        "чанков": len(ch),
        "чанков/док (среднее)": round(ch.groupby("doc_id").size().mean(), 2),
    })
display(pd.DataFrame(stats))

ex = chunks_py.iloc[0][["chunk_id", "doc_id", "title", "authors", "year", "url", "text"]]
display(ex.to_frame("пример чанка"))

,способ,чанков,чанков/док (среднее)
0,чистый Python,924,1.94
1,LangChain,1410,2.96


,пример чанка
chunk_id,2001.01895::py::0
doc_id,2001.01895
title,Machine-learning classifiers for logographic name matching in public health applications: approaches for i...
authors,Philip A. Collender; Zhiyue Tom Hu; Charles Li; Qu Cheng; Xintong Li; Yue You; Song Liang; Changhong Yang;...
year,2020
url,https://arxiv.org/abs/2001.01895
text,Machine-learning classifiers for logographic name matching in public health applications: approaches for i...


**Вывод по пункту 2.** Скользящее окно даёт 924 чанка. RecursiveCharacterTextSplitter — 1410 чанков

## Пункт 3. Векторизация и индекс (Vector Store)

> Инициализировать легковесную модель для построения эмбеддингов предложений. Перевести все чанки в плотные векторы. Построить векторный индекс с использованием FAISS (IndexFlatIP с предварительной L2-нормализацией векторов для косинусного сходства). Реализовать функцию retrieve(query, top_k), возвращающую top_k наиболее релевантных фрагментов с оценками сходства. Все шаги выполняются для каждого способа чанкинга по отдельности.

Модель `all-MiniLM-L6-v2` превращает каждый фрагмент в 384-мерный вектор. Векторы L2-нормализуются, поэтому индекс `IndexFlatIP` (скалярное произведение) вычисляет косинусное сходство. Для каждого способа чанкинга строится свой индекс.

In [31]:
EMB_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
emb_model = SentenceTransformer(EMB_MODEL_NAME)


def encode_texts(texts):
    emb = emb_model.encode(texts, batch_size=64, normalize_embeddings=True)
    return np.ascontiguousarray(emb.astype("float32"))


emb_py = encode_texts(chunks_py["text"].tolist())
emb_lc = encode_texts(chunks_lc["text"].tolist())


def build_faiss_index(emb):
    emb = emb.copy()
    faiss.normalize_L2(emb)
    index = faiss.IndexFlatIP(emb.shape[1])
    index.add(emb)
    return index


index_py = build_faiss_index(emb_py)
index_lc = build_faiss_index(emb_lc)
INDEXES = {"py": index_py, "lc": index_lc}
TABLES = {"py": chunks_py, "lc": chunks_lc}

pd.DataFrame({"чанкинг": ["Python", "LangChain"],
                  "эмбеддинги": [str(emb_py.shape), str(emb_lc.shape)]})

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10333.76it/s]


,чанкинг,эмбеддинги
0,Python,"(924, 384)"
1,LangChain,"(1410, 384)"


Функция `retrieve(query, top_k)` возвращает фрагменты с оценками сходства и метаданными родительской статьи. Пробуем один запрос на обоих индексах.

In [32]:
def retrieve(query, top_k=5, method="py"):
    q_emb = emb_model.encode([query], normalize_embeddings=True).astype("float32")
    scores, ids = INDEXES[method].search(q_emb, top_k)
    rows = []
    for score, idx in zip(scores[0], ids[0]):
        r = TABLES[method].iloc[int(idx)]
        rows.append({"score": float(score), **{k: r[k] for k in
                    ["chunk_id", "doc_id", "title", "authors", "year", "url", "text"]}})
    return pd.DataFrame(rows)


RAG_METHOD, RAG_METHOD_LABEL = "lc", "LangChain"

demo_query = "Which blocking techniques make entity resolution scalable?"
demo_rows = []
for method, label in [("py", "чистый Python"), ("lc", "LangChain")]:
    for _, r in retrieve(demo_query, top_k=3, method=method).iterrows():
        demo_rows.append({"чанкинг": label, "score": round(r["score"], 4),
                          "doc_id": r["doc_id"], "title": r["title"][:80], "year": r["year"]})
pd.DataFrame(demo_rows)

,чанкинг,score,doc_id,title,year
0,чистый Python,0.7475,2005.14326,Efficient and Effective ER with Progressive Blocking,2020
1,чистый Python,0.6705,2204.08801,Generalized Supervised Meta-blocking (technical report),2022
2,чистый Python,0.6631,2202.12521,How to reduce the search space of Entity Resolution: with Blocking or Nearest Ne,2022
3,LangChain,0.7728,2202.12521,How to reduce the search space of Entity Resolution: with Blocking or Nearest Ne,2022
4,LangChain,0.7712,2005.14326,Efficient and Effective ER with Progressive Blocking,2020
5,LangChain,0.7607,2504.04266,BlockingPy: approximate nearest neighbours for blocking of records for entity re,2025


**Вывод по пункту 3.** Оба индекса построены и работают: на запрос о техниках блокировки каждый возвращает статьи о блокировке Entity Resolution с косинусным сходством до 0.75. Дальше в конвейере используется индекс чанкинга LangChain.

## Пункт 4. Обработка пользовательских запросов

> Пользовательский ввод может содержать случайные искажения. Реализовать пайплайн очистки запроса: проверка и коррекция опечаток с использованием SymSpell или rapidfuzz (с осторожностью к аббревиатурам и научным терминам); формирование чистого поискового запроса для поиска по индексу.

Исправляем только то, что похоже на опечатку, а не на термин. В защиту попадают лишь те частотные слова корпуса (не менее 25 вхождений), которые SymSpell исправил бы во что-то другое, плюс четыре редких термина вручную (`deduplication`, `minhash`, `dedupe`, `matchers`) — без этого `minhash` превращается в `midrash`

In [33]:
sym = SymSpell(max_dictionary_edit_distance=2, prefix_length=7)
DICT_PATH = Path(symspellpy.__file__).parent / "frequency_dictionary_en_82_765.txt"
sym.load_dictionary(str(DICT_PATH), 0, 1)

corpus_words = Counter(w.lower() for t in docs["text"] for w in re.findall(r"[A-Za-z][A-Za-z'-]{3,}", t))


def is_vulnerable(word):
    sug = sym.lookup(word, Verbosity.TOP, 2)
    return bool(sug) and sug[0].term != word and sug[0].count >= 5000


DOMAIN_TERMS = {w for w, c in corpus_words.items() if c >= 25 and is_vulnerable(w)} | {
    "deduplication", "minhash", "dedupe", "matchers",
}
for term in DOMAIN_TERMS:
    sym.create_dictionary_entry(term, 10_000_000)

WORD_RE = re.compile(r"[A-Za-z][A-Za-z'-]*")


def clean_query(query, min_count=5000):
    out = []
    for tok in re.findall(r"[A-Za-z][A-Za-z'-]*|\S+", query):
        if not WORD_RE.fullmatch(tok):
            out.append(tok)
            continue
        low = tok.lower()
        if low in DOMAIN_TERMS or len(low) < 4 or (tok.isupper() and len(tok) <= 6):
            out.append(tok)
            continue
        sug = sym.lookup(low, Verbosity.TOP, 2)
        if sug and sug[0].distance > 0 and sug[0].count >= min_count:
            rep = sug[0].term
            out.append(rep.capitalize() if tok[:1].isupper() else rep)
        else:
            out.append(tok)
    text = " ".join(out)
    return re.sub(r"\s+([,.;:!?])", r"\1", text)


typo_tests = [
    "entity resolutoin metods for large scale datbases",
    "blokcing stratagies to acelerate entity matchng",
    "deep lerning vs rule based entity resoltuion",
    "LSH based blocking for ER with MinHash signatres",
    "author name disambiguaton in bibliografic databases",
    "record linkage with probabilistc models",
]
pd.DataFrame({"исходный запрос": typo_tests,
              "после очистки": [clean_query(q) for q in typo_tests]})

,исходный запрос,после очистки
0,entity resolutoin metods for large scale datbases,entity resolution methods for large scale databases
1,blokcing stratagies to acelerate entity matchng,blocking strategies to accelerate entity matching
2,deep lerning vs rule based entity resoltuion,deep learning vs rule based entity resolution
3,LSH based blocking for ER with MinHash signatres,LSH based blocking for ER with MinHash signatures
4,author name disambiguaton in bibliografic databases,author name disambiguation in bibliographic databases
5,record linkage with probabilistc models,record linkage with probabilistic models


In [34]:
def retrieve_pipeline(query, top_k=5, method="py"):
    cleaned = clean_query(query)
    return cleaned, retrieve(cleaned, top_k=top_k, method=method)


raw_q = "entity resolutoin metods for large scale datbases"
cleaned_q, res_q = retrieve_pipeline(raw_q, top_k=3, method=RAG_METHOD)
display(pd.DataFrame({"исходный запрос": [raw_q], "поисковый запрос": [cleaned_q]}))
res_q[["score", "doc_id", "title", "year"]]

,исходный запрос,поисковый запрос
0,entity resolutoin metods for large scale datbases,entity resolution methods for large scale databases


,score,doc_id,title,year
0,0.786530,2509.17470,"Transformer-Gather, Fuzzy-Reconsider: A Scalable Hybrid Framework for Entity Resolution",2025
1,0.749046,2209.07569,FlexER: Flexible Entity Resolution for Multiple Intents,2022
2,0.747620,2508.03767,A Robust and Efficient Pipeline for Enterprise-Level Large-Scale Entity Resolution,2025


## Пункт 5. Интеграция с LLM и промпт-инжиниринг

> Настроить компактную LLM для синтеза ответа по найденному контексту:
> - Локально в Colab/ноутбуке (например, Qwen/Qwen2.5-1.5B-Instruct, meta-llama/Llama-3.2-1B-Instruct через transformers / bitsandbytes 4-bit);
> - Либо через API (OpenAI GPT-4o-mini, Mistral API, Groq, DeepSeek).
>
> Требования к промпту:
> - Модель должна отвечать строго на основе предоставленного контекста;
> - Если в контексте нет ответа, модель обязана явно заявить: «Информация в найденных статьях отсутствует»;
> - Модель должна указывать ссылки/названия статей, на основе которых сформулирован вывод (Grounding / Citations).
>
> Составить 5 различных сценариев запросов:
> - Фактический вопрос по методикам;
> - Сравнительный анализ двух подходов;
> - Запрос с опечатками;
> - Запрос, на который в корпусе нет ответа (проверка на галлюцинации);
> - Обобщающий запрос (Summarization) по ключевым трендам в собранных статьях.
>
> Наглядно продемонстрировать разницу: ответ без контекста (чистая LLM) vs ответ с RAG (LLM + контекст из базы).

In [35]:
GLM_MODEL = "glm-5.3"
glm_client = Anthropic(api_key=GLM_API_KEY, base_url="https://api.z.ai/api/anthropic")


def ask_glm(system, user, max_tokens=12000, retries=3):
    last_error = None
    for attempt in range(retries):
        try:
            resp = glm_client.messages.create(
                model=GLM_MODEL, max_tokens=max_tokens, system=system,
                messages=[{"role": "user", "content": user}],
            )
            text = "".join(b.text for b in resp.content if getattr(b, "type", "") == "text").strip()
            if text:
                u = resp.usage
                return text, {"in": u.input_tokens, "out": u.output_tokens}
            last_error = f"пустой ответ: {[getattr(b, 'type', '?') for b in resp.content]}"
        except Exception as e:
            last_error = f"{type(e).__name__}: {str(e)[:200]}"
        time.sleep(3 * (attempt + 1))
    raise RuntimeError(f"GLM API недоступен: {last_error}")


SYSTEM_PROMPT_RAG = """Ты - научный ассистент по базе англоязычных статей arXiv по теме Entity Resolution.
Отвечай на вопрос пользователя СТРОГО на основе предоставленного контекста - фрагментов статей с номерами [1], [2], ...

Правила:
1. Используй ТОЛЬКО факты из контекста. Не добавляй собственные знания и не домысливай.
2. Если ответа на вопрос в контексте нет - ответь ровно одной фразой: Информация в найденных статьях отсутствует
   (и ничего больше).
3. Каждое утверждение подкрепляй ссылкой на номер источника в квадратных скобках, например [1] или [2, 3].
4. В конце ответа приведи раздел "Источники": список использованных статей в формате [n] Название (URL).
5. Отвечай по-русски, специальные термины можно оставлять на английском."""


def build_context(res):
    blocks = []
    for i, r in enumerate(res.itertuples(index=False), 1):
        first_author = r.authors.split(";")[0].strip()
        blocks.append(f"[{i}] {r.title} ({first_author} и др., {r.year}). {r.url}\n{r.text}")
    return "\n\n".join(blocks)


CYR_RE = re.compile(r"[А-Яа-яЁё]")


def translate_to_english(text):
    translation, _ = ask_glm("Ты - переводчик. Переведи текст на английский язык. Выведи только перевод.", text)
    return translation.strip()


def rag_answer(query, top_k=5, method=RAG_METHOD):
    search_query = translate_to_english(query) if CYR_RE.search(query) else query
    cleaned, res = retrieve_pipeline(search_query, top_k=top_k, method=method)
    user_msg = f"Контекст - фрагменты статей:\n\n{build_context(res)}\n\nВопрос: {query}"
    answer, usage = ask_glm(SYSTEM_PROMPT_RAG, user_msg)
    return cleaned, res, answer, usage


def llm_no_context(query):
    system = "Ты - полезный ассистент. Отвечай кратко, по делу, на языке вопроса."
    answer, usage = ask_glm(system, query)
    return answer, usage

print(SYSTEM_PROMPT_RAG)

Ты - научный ассистент по базе англоязычных статей arXiv по теме Entity Resolution.
Отвечай на вопрос пользователя СТРОГО на основе предоставленного контекста - фрагментов статей с номерами [1], [2], ...

Правила:
1. Используй ТОЛЬКО факты из контекста. Не добавляй собственные знания и не домысливай.
2. Если ответа на вопрос в контексте нет - ответь ровно одной фразой: Информация в найденных статьях отсутствует
   (и ничего больше).
3. Каждое утверждение подкрепляй ссылкой на номер источника в квадратных скобках, например [1] или [2, 3].
4. В конце ответа приведи раздел "Источники": список использованных статей в формате [n] Название (URL).
5. Отвечай по-русски, специальные термины можно оставлять на английском.


In [36]:
SCENARIO_LOG = []


def show_scenario(query, top_k=5):
    print("=" * 100)
    print(f"Запрос: {query}")
    cleaned, res, answer_rag, u_rag = rag_answer(query, top_k=top_k)
    answer_llm, u_llm = llm_no_context(query)
    print(f"Поисковый запрос: {cleaned}")
    print(f"\nНайденные фрагменты (top-{top_k}, чанкинг {RAG_METHOD_LABEL}):")
    for i, r in enumerate(res.itertuples(index=False), 1):
        print(f"  [{i}] score={r.score:.4f}  [{r.doc_id}] {r.title[:82]} ({r.year})")
    print(f"\n{'-' * 100}\nОТВЕТ LLM БЕЗ КОНТЕКСТА (GLM 5.3, собственные знания; токены in/out: {u_llm['in']}/{u_llm['out']}):\n")
    print(answer_llm)
    print(f"\n{'-' * 100}\nОТВЕТ RAG (GLM 5.3 + контекст из {top_k} фрагментов; токены in/out: {u_rag['in']}/{u_rag['out']}):\n")
    print(answer_rag)
    cited = sorted({int(n) for n in re.findall(r"\[(\d+)\]", answer_rag)})
    SCENARIO_LOG.append({
        "запрос": query, "цитирует [n]": cited,
        "отказ (нет информации)": answer_rag.strip() == "Информация в найденных статьях отсутствует",
        "_answer": answer_rag, "_res": res, "_cited": cited,
    })

**Сценарий 1 — фактический вопрос.**

In [37]:
show_scenario("Какие техники блокировки и фильтрации используются, чтобы сделать Entity Resolution масштабируемым на больших данных?")

Запрос: Какие техники блокировки и фильтрации используются, чтобы сделать Entity Resolution масштабируемым на больших данных?
Поисковый запрос: What blocking and filtering techniques are used to make Entity Resolution scalable on large datasets?

Найденные фрагменты (top-5, чанкинг LangChain):
  [1] score=0.7901  [2202.12521] How to reduce the search space of Entity Resolution: with Blocking or Nearest Neig (2022)
  [2] score=0.7497  [2303.03132] SC-Block: Supervised Contrastive Blocking within Entity Resolution Pipelines (2023)
  [3] score=0.7419  [2504.04266] BlockingPy: approximate nearest neighbours for blocking of records for entity reso (2025)
  [4] score=0.7152  [2005.14326] Efficient and Effective ER with Progressive Blocking (2020)
  [5] score=0.6967  [2005.09399] Benchmarking Blocking Algorithms for Web Entities (2020)

----------------------------------------------------------------------------------------------------
ОТВЕТ LLM БЕЗ КОНТЕКСТА (GLM 5.3, собственные знания; ток

**Сценарий 2 — сравнительный анализ.**

In [38]:
show_scenario("Сравни сопоставление сущностей на основе глубокого обучения и традиционные подходы на правилах: точность, потребность в обучающих данных, интерпретируемость.")

Запрос: Сравни сопоставление сущностей на основе глубокого обучения и традиционные подходы на правилах: точность, потребность в обучающих данных, интерпретируемость.
Поисковый запрос: Compare deep learning-based entity matching and traditional rule-based approaches: accuracy, training data requirements, and interpretability.

Найденные фрагменты (top-5, чанкинг LangChain):
  [1] score=0.7572  [2010.11075] Neural Networks for Entity Matching: A Survey (2020)
  [2] score=0.7545  [2010.11075] Neural Networks for Entity Matching: A Survey (2020)
  [3] score=0.7151  [2307.01231] A Critical Re-evaluation of Benchmark Datasets for (Deep) Learning-Based Matching  (2023)
  [4] score=0.7039  [2205.05889] Bridging the Gap between Reality and Ideality of Entity Matching: A Revisiting and (2022)
  [5] score=0.6982  [2203.12978] Effective Explanations for Entity Resolution Models (2022)

----------------------------------------------------------------------------------------------------
ОТВЕТ LLM БЕ

**Сценарий 3 — запрос с опечатками.**

In [39]:
show_scenario("методы разрешения сущнастей для больших баз даных")

Запрос: методы разрешения сущнастей для больших баз даных
Поисковый запрос: entity resolution methods for large databases

Найденные фрагменты (top-5, чанкинг LangChain):
  [1] score=0.7725  [2509.17470] Transformer-Gather, Fuzzy-Reconsider: A Scalable Hybrid Framework for Entity Resol (2025)
  [2] score=0.7426  [2209.07569] FlexER: Flexible Entity Resolution for Multiple Intents (2022)
  [3] score=0.7375  [2512.23491] SPER: Accelerating Progressive Entity Resolution via Stochastic Bipartite Maximiza (2025)
  [4] score=0.7318  [2503.08087] Resolvi: A Reference Architecture for Extensible, Scalable and Interoperable Entit (2025)
  [5] score=0.7273  [2303.06629] Another Generic Setting for Entity Resolution: Basic Theory (2023)

----------------------------------------------------------------------------------------------------
ОТВЕТ LLM БЕЗ КОНТЕКСТА (GLM 5.3, собственные знания; токены in/out: 49/892):

# Методы разрешения сущностей (Entity Resolution) для больших БД

## 1. Блокировка 

**Сценарий 4 — запрос без ответа в корпусе.**

In [40]:
show_scenario("Какие сверхпроводящие материалы используются для создания кубитов в квантовых компьютерах?")

Запрос: Какие сверхпроводящие материалы используются для создания кубитов в квантовых компьютерах?
Поисковый запрос: What superconducting materials are used to create quits in quantum computers?

Найденные фрагменты (top-5, чанкинг LangChain):
  [1] score=0.2755  [2503.05596] Bridging Classical and Quantum String Matching: A Computational Reformulation of B (2025)
  [2] score=0.2543  [2605.04388] Hyperspectral Anomaly Detection Using Einstein Fuzzy Computing and Quantum Neural  (2026)
  [3] score=0.2174  [2605.04388] Hyperspectral Anomaly Detection Using Einstein Fuzzy Computing and Quantum Neural  (2026)
  [4] score=0.2069  [2302.08284] ClaPIM: Scalable Sequence CLAssification using Processing-In-Memory (2023)
  [5] score=0.2001  [2503.05596] Bridging Classical and Quantum String Matching: A Computational Reformulation of B (2025)

----------------------------------------------------------------------------------------------------
ОТВЕТ LLM БЕЗ КОНТЕКСТА (GLM 5.3, собственные знания; 

**Сценарий 5 — обобщающий запрос.** 

In [41]:
show_scenario("Какие основные тренды в исследованиях Entity Resolution видны по статьям корпуса: преобладающие методы, задачи и практика оценки качества?", top_k=8)

Запрос: Какие основные тренды в исследованиях Entity Resolution видны по статьям корпуса: преобладающие методы, задачи и практика оценки качества?
Поисковый запрос: What are the main trends in Entity Resolution research visible from the corpus papers: prevailing methods, tasks, and quality evaluation practices?

Найденные фрагменты (top-8, чанкинг LangChain):
  [1] score=0.6954  [2004.02008] Random Partition Models for Microclustering Tasks (2020)
  [2] score=0.6561  [2602.05452] DistillER: Knowledge Distillation in Entity Resolution with Large Language Models (2026)
  [3] score=0.6497  [2506.13839] Towards an Approach for Evaluating the Impact of AI Standards The use case of enti (2025)
  [4] score=0.6482  [2503.08087] Resolvi: A Reference Architecture for Extensible, Scalable and Interoperable Entit (2025)
  [5] score=0.6443  [2607.27435] AgenticER: the next frontier in Entity Resolution (2026)
  [6] score=0.6387  [2506.02509] In-context Clustering-based Entity Resolution with Large 

**Вывод по пункту 5.**

- **Сценарии 1–2.** Чистая LLM отвечает правдоподобно, но общими словами и без опоры на конкретные работы. RAG отвечает по найденным статьям со ссылками [n] и прямо отмечает, какие части вопроса во фрагментах не раскрыты, а не додумывает их.
- **Сценарий 3.** Русский запрос с опечатками переведён в корректный английский поисковый запрос, retrieval нашёл статьи о масштабируемом Entity Resolution — конвейер устойчив к искажениям ввода.
- **Сценарий 4.** LLM без контекста уверенно перечисляет сверхпроводники для кубитов (алюминий, ниобий, тантал). RAG ответил ровно: «Информация в найденных статьях отсутствует».
- **Сценарий 5.** RAG обобщает тренды корпуса со ссылками на 8 статей: эволюция от rule-based и deep learning к LLM, потребность в размеченных данных, гибридные методы.

**Итог.** Собран RAG-конвейер над корпусом из 477 статей по Entity Resolution. Документы «название + аннотация» разбиты на чанки двумя способами: 924 чанка на чистом Python (окно 150 слов, перекрытие 25) и 1410 чанков через LangChain (окно 650 символов, перекрытие 100); каждый чанк несёт метаданные статьи. Чанки превращены в эмбеддинги all-MiniLM-L6-v2 и разложены по двум индексам FAISS. Запросы принимаются на русском: GLM переводит их на английский, SymSpell чистит опечатки, не трогая термины и аббревиатуры. Ответы генерирует GLM 5.3: RAG отвечает строго по корпусу со ссылками на источники и отказывается отвечать на вопросы вне корпуса, тогда как та же модель без контекста отвечает по памяти.